In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, T5EncoderModel, AutoModelForTextEncoding
from sentence_transformers import SentenceTransformer

tokenizer = AutoTokenizer.from_pretrained("ai-forever/FRIDA")
model = SentenceTransformer("ai-forever/FRIDA")



Loading weights: 100%|██████████| 219/219 [00:00<00:00, 893.41it/s]


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


In [4]:
def predict(texts, batch_size=4):
    texts = [f"categorize: {t}" for t in texts]
    text_emb = encode_batch(texts, batch_size=batch_size)
    scores = text_emb @ label_emb.T
    probs = torch.sigmoid(scores)
    return probs

In [5]:
import pandas as pd

In [6]:
df=pd.read_csv("new_ds.csv")
df

,text,ASSORTMENT,PROMOTIONS,DELIVERY,PRICE,PRODUCTS_QUALITY,SUPPORT,CATALOG_NAVIGATION,PAYMENT
0,"Маленький выбор товаров, хотелось бы ассортиме...",1,0,0,0,0,0,0,0
1,Быстро,0,0,1,0,0,0,0,0
2,Доставка постоянно задерживается,0,0,1,0,0,0,0,0
3,Наценка и ассортимент расстраивают,1,0,0,1,0,0,0,0
4,Можно немного скинуть минимальную сумму заказа...,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...
2277,"Очень расстраивает, что сервис стал постоянно ...",0,0,1,0,0,0,0,0
2278,Хлеб привезли не свежий,0,0,0,0,1,0,0,0
2279,"Сервис испортился,доставка по [NUM],[NUM] часа(",0,0,1,0,0,0,0,0
2280,Последнее время постоянные проблемы с доставко...,0,0,1,0,0,0,0,0


In [7]:
label_texts = [
    "category: проблемы с доставкой, долгая доставка, курьер",
    "category: акции, скидки, промокоды",
    "category: ассортимент товаров, выбор",
    "category: высокая или низкая цена",
    "category: качество товара, брак",
    "category: служба поддержки, помощь клиенту",
    "category: навигация по каталогу, поиск товаров",
    "category: оплата, способы оплаты, проблемы с оплатой"
]

In [8]:
def pool(hidden_state, mask):
    s = torch.sum(hidden_state * mask.unsqueeze(-1).float(), dim=1)
    d = mask.sum(axis=1, keepdim=True).float()
    return s / d

def encode_batch(texts, batch_size=8):
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokens = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**tokens)
        emb = pool(outputs.last_hidden_state, tokens["attention_mask"])
        embs.append(F.normalize(emb, p=2, dim=1))
    return torch.cat(embs, dim=0).cpu().numpy()  # на CPU для sklearn

In [13]:
text=df["text"]
y_true=df.drop(columns=["text"])

In [11]:
from sklearn.model_selection import train_test_split


In [14]:
X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    text, y_true, test_size=0.2, random_state=42,shuffle=True
)

X_train_texts = X_train_texts.astype(str).tolist()
X_val_texts = X_val_texts.astype(str).tolist()



In [ ]:
X_train = encode_batch(X_train_texts)
X_val = encode_batch(X_val_texts)


In [28]:
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig

In [2]:
model = SentenceTransformer(
    "ai-forever/FRIDA",
    model_kwargs={"torch_dtype": "float16"}
)

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 1731.30it/s]


In [3]:
#model = SentenceTransformer("ai-forever/FRIDA", model_kwargs={"load_in_8bit": True})

In [15]:
X_train = model.encode(X_train_texts, batch_size=8, show_progress_bar=True)
X_val = model.encode(X_val_texts, batch_size=8, show_progress_bar=True)

Batches: 100%|██████████| 58/58 [00:01<00:00, 29.76it/s]


In [ ]:
import lightgbm as lgb
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score



base_clf = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    class_weight='balanced',  # сбалансирует 0 и 1 автоматически
    random_state=42,
    verbosity=-1
)
#{'n_estimators': 997, 'learning_rate': 0.23919409346136555, 'max_depth': 3, 'num_leaves': 87, 'min_child_samples': 81}
clf = MultiOutputClassifier(base_clf)
# обучаем
clf.fit(X_train, y_train)

# предсказания
y_pred = clf.predict(X_val)


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [28]:
import optuna
import lightgbm as lgb
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score

def objective(trial):
    # 1. Определяем пространство поиска гиперпараметров
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'class_weight': 'balanced',
        'random_state': 42,
        'verbosity': -1
    }

    # 2. Создаем базовый классификатор с предложенными параметрами
    base_clf = lgb.LGBMClassifier(**param)
    
    # 3. Оборачиваем его в MultiOutput
    model = MultiOutputClassifier(base_clf)
    
    # 4. Обучаем
    model.fit(X_train, y_train)
    
    # 5. Делаем предсказание и считаем метрику
    y_pred = model.predict(X_val)
    
    # Для мультилейбл классификации обычно используют 'macro' или 'weighted' f1
    score = f1_score(y_val, y_pred, average='macro')
    
    return score

# 6. Запуск процесса оптимизации
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"Лучший F1-score: {study.best_value}")
print(f"Лучшие параметры: {study.best_params}")

[I 2026-04-15 23:24:11,972] A new study created in memory with name: no-name-c8317d8d-8d46-48ca-8195-e76b12c84d7e
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/h

Лучший F1-score: 0.683518275787535
Лучшие параметры: {'n_estimators': 997, 'learning_rate': 0.23919409346136555, 'max_depth': 3, 'num_leaves': 87, 'min_child_samples': 81}


In [12]:
y_val

,ASSORTMENT,PROMOTIONS,DELIVERY,PRICE,PRODUCTS_QUALITY,SUPPORT,CATALOG_NAVIGATION,PAYMENT
1268,0,0,0,0,0,0,1,0
1633,0,0,1,0,0,1,0,0
700,0,0,1,0,1,1,0,0
2028,0,0,1,0,0,0,0,0
596,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...
387,0,0,1,0,0,0,0,0
649,0,0,0,0,1,0,0,0
1645,0,0,0,0,0,0,0,0
1318,0,0,1,0,1,0,1,0


In [30]:
import numpy as np

y_probs = np.array([est.predict_proba(X_val)[:, 1] for est in clf.estimators_]).T


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [18]:
import numpy as np

In [33]:
y_pred_proba = clf.predict_proba(X_val)  # список массивов для каждой метки
y_pred_thresh = np.zeros_like(y_val)

thresholds = [0.2, 0.0144595, 0.28073387, 0.04927171, 0.14049791, 0.01041178, 0.00265701, 0.00237321]  # подбираются эмпирически

for i, th in enumerate(thresholds):
    y_pred_thresh[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)

f1 = f1_score(y_val, y_pred_thresh, average='macro')
print(f"Macro F1 с порогами: {f1:.3f}")

Macro F1 с порогами: 0.722


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [32]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score
import numpy as np



def macro_f1(thresholds):
    y_pred = np.zeros_like(y_val)
    for i, th in enumerate(thresholds):
        y_pred[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)
    return -f1_score(y_val, y_pred, average='macro') 

bounds = [(0, 1)] * len(y_pred_proba)
result = differential_evolution(macro_f1, bounds)
optimal_thresholds = result.x

# Применяем
y_pred_thresh = np.zeros_like(y_val)
for i, th in enumerate(optimal_thresholds):
    y_pred_thresh[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)

f1 = f1_score(y_val, y_pred_thresh, average='macro')
print("Максимальный Macro F1:", f1)
print("Порог для каждой метки:", optimal_thresholds)

Максимальный Macro F1: 0.7199643345411006
Порог для каждой метки: [0.02535451 0.00910065 0.57046175 0.85250561 0.54364808 0.6986579
 0.07009778 0.00577671]


In [34]:
from sklearn.metrics import classification_report
print(classification_report(y_val,y_pred_thresh))

              precision    recall  f1-score   support

           0       0.87      0.80      0.84        41
           1       0.91      0.71      0.80        14
           2       0.92      0.93      0.92       243
           3       0.88      0.88      0.88        84
           4       0.88      0.91      0.89        93
           5       0.75      0.67      0.70        57
           6       0.30      0.26      0.28        34
           7       0.75      0.33      0.46         9

   micro avg       0.85      0.83      0.84       575
   macro avg       0.78      0.69      0.72       575
weighted avg       0.85      0.83      0.84       575
 samples avg       0.73      0.72      0.70       575



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [35]:
from sklearn.metrics import hamming_loss

hloss = hamming_loss(y_val, y_pred_thresh)
print("Hamming Loss:", hloss)

Hamming Loss: 0.04950765864332604


In [19]:
from sklearn.metrics import classification_report
print(classification_report(y_val,y_pred_thresh))

              precision    recall  f1-score   support

           0       0.92      0.83      0.87        41
           1       0.90      0.64      0.75        14
           2       0.97      0.89      0.93       243
           3       0.91      0.86      0.88        84
           4       0.93      0.87      0.90        93
           5       0.81      0.61      0.70        57
           6       0.32      0.32      0.32        34
           7       1.00      0.33      0.50         9

   micro avg       0.90      0.80      0.85       575
   macro avg       0.85      0.67      0.73       575
weighted avg       0.90      0.80      0.84       575
 samples avg       0.73      0.69      0.69       575



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [25]:
new_text_features = model.encode(["Доставка опоздала и продукты испортились"])

new_probs = clf.predict_proba(new_text_features)
new_probs

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

[array([[9.99996474e-01, 3.52575934e-06]]),
 array([[9.99999849e-01, 1.51049256e-07]]),
 array([[0.00381363, 0.99618637]]),
 array([[9.99996153e-01, 3.84711718e-06]]),
 array([[7.71059461e-05, 9.99922894e-01]]),
 array([[9.99997624e-01, 2.37630264e-06]]),
 array([[9.99999472e-01, 5.28382360e-07]]),
 array([[9.99998917e-01, 1.08328789e-06]])]

In [26]:

new_pred = np.zeros((new_text_features.shape[0], len(thresholds)), dtype=int)
for i, th in enumerate(thresholds):
    new_pred[:, i] = (new_probs[i][:, 1] >= th).astype(int)

print(new_pred)

[[0 0 1 0 1 0 0 0]]


In [21]:
import joblib
import json
import os

In [22]:
os.makedirs("model_weights", exist_ok=True)

In [23]:
model.save_pretrained("./model_weights/enc")

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.29s/it]


In [23]:
tokenizer.save_pretrained("./model_weights/tokenizer")
model.save_pretrained("./model_weights/encoder")

Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.98s/it]


In [24]:
joblib.dump(clf, "./model_weights/classifier.joblib")

['./model_weights/classifier.joblib']

In [25]:
thresholds

[0.2,
 0.0144595,
 0.28073387,
 0.04927171,
 0.14049791,
 0.01041178,
 0.00265701,
 0.00237321]

In [26]:
with open("./model_weights/thresholds.json", "w") as f:
    json.dump(thresholds, f)

In [27]:
t5=T5EncoderModel.from_pretrained("model_weights/encoder")

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 13759.23it/s]


In [28]:
tokeniz=AutoTokenizer.from_pretrained("model_weights/encoder")

In [3]:
from transformers import AutoModelForSequenceClassification

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model=AutoModelForSequenceClassification.from_pretrained("ai-forever/FRIDA",num_labels=8)

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 1094.62it/s]
T5ForSequenceClassification LOAD REPORT from: ai-forever/FRIDA
Key                                                                              | Status  | 
---------------------------------------------------------------------------------+---------+-
transformer.decoder.block.{0...23}.layer.2.DenseReluDense.wo.weight              | MISSING | 
transformer.decoder.block.{0...23}.layer.1.EncDecAttention.k.weight              | MISSING | 
transformer.decoder.block.{0...23}.layer.{0, 1, 2}.layer_norm.weight             | MISSING | 
transformer.decoder.block.{0...23}.layer.0.SelfAttention.k.weight                | MISSING | 
transformer.decoder.block.{0...23}.layer.1.EncDecAttention.q.weight              | MISSING | 
transformer.decoder.block.{0...23}.layer.0.SelfAttention.v.weight                | MISSING | 
transformer.decoder.block.{0...23}.layer.2.DenseReluDense.wi_0.weight            | MISSING | 
transformer.decoder.bl

In [5]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("ai-forever/FRIDA")

In [6]:
tokens=tokenizer(df["text"].to_list(),padding="max_length",truncation=True,max_length=1024,return_tensors="pt")

In [7]:
tokens

{'input_ids': tensor([[    1, 22052,  8908,  ...,     0,     0,     0],
        [    1, 10934,   884,  ...,     0,     0,     0],
        [    1, 12206, 11003,  ...,     0,     0,     0],
        ...,
        [    1, 14726,  1589,  ...,     0,     0,     0],
        [    1, 18075,   912,  ...,     0,     0,     0],
        [    1, 12206, 11003,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [9]:
import torch
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() # Перевод в режим оценки

batch_size = 16 
texts = df["text"].to_list()
all_embeddings = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        
        # 1. Токенизируем текущий батч
        inputs = tokenizer(
            batch_texts, 
            padding=True, 
            truncation=True, 
            max_length=1024, 
            return_tensors="pt"
        )
        
        # 2. ПЕРЕНОСИМ ДАННЫЕ НА GPU (Критически важно!)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 3. Прогоняем через модель
        outputs = model(**inputs)
        
        # 4. Получаем логиты и отправляем обратно на CPU, чтобы не забивать VRAM
        logits = outputs.logits.cpu() 
        all_embeddings.append(logits)

# Объединяем всё в один тензор/массив в конце
final_embeddings = torch.cat(all_embeddings, dim=0)
print(final_embeddings.shape)

  0%|          | 0/143 [00:00<?, ?it/s]

100%|██████████| 143/143 [01:17<00:00,  1.84it/s]

torch.Size([2282, 8])


tensor([[ 1.0895, -0.6831, -0.0155,  ..., -0.0299, -0.5911, -0.1820],
        [-1.5820,  0.7590,  0.6198,  ...,  1.5049,  0.3855,  0.0961],
        [-0.0086, -0.0459, -0.2378,  ...,  0.5180,  1.0020,  0.4713],
        ...,
        [ 0.3301,  0.7960, -0.1338,  ..., -0.1107,  0.1603, -0.4121],
        [ 0.1879,  0.1995,  0.4825,  ..., -0.7059, -0.4190, -1.0547],
        [-0.2573, -0.5094, -0.6290,  ..., -0.8364, -0.1495, -0.1636]])

In [11]:
from sklearn.preprocessing import MultiLabelBinarizer
import torch

mlb = MultiLabelBinarizer()
labels_binary = mlb.fit_transform(df.drop(columns=["text"])) 
target_tensor = torch.tensor(labels_binary, dtype=torch.float32)

,ASSORTMENT,PROMOTIONS,DELIVERY,PRICE,PRODUCTS_QUALITY,SUPPORT,CATALOG_NAVIGATION,PAYMENT
0,1,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0
2,0,0,1,0,0,0,0,0
3,1,0,0,1,0,0,0,0
4,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...
2277,0,0,1,0,0,0,0,0
2278,0,0,0,0,1,0,0,0
2279,0,0,1,0,0,0,0,0
2280,0,0,1,0,0,0,0,0


In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    "ai-forever/FRIDA", 
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 14297.65it/s]
T5ForSequenceClassification LOAD REPORT from: ai-forever/FRIDA
Key                                                                              | Status  | 
---------------------------------------------------------------------------------+---------+-
transformer.decoder.block.{0...23}.layer.2.DenseReluDense.wo.weight              | MISSING | 
transformer.decoder.block.{0...23}.layer.1.EncDecAttention.k.weight              | MISSING | 
transformer.decoder.block.{0...23}.layer.{0, 1, 2}.layer_norm.weight             | MISSING | 
transformer.decoder.block.{0...23}.layer.0.SelfAttention.k.weight                | MISSING | 
transformer.decoder.block.{0...23}.layer.1.EncDecAttention.q.weight              | MISSING | 
transformer.decoder.block.{0...23}.layer.0.SelfAttention.v.weight                | MISSING | 
transformer.decoder.block.{0...23}.layer.2.DenseReluDense.wi_0.weight            | MISSING | 
transformer.decoder.b

In [ ]:
tokeniz